In [2]:
import os
import datetime
from openai import OpenAI
import anthropic
from typing import Optional
from tqdm import tqdm
import pandas as pd


In [3]:
### EDIT THESE EVERY NEW STORY ###


MODEL = "ft:gpt-5-2024-07-18:personal::AwdUXsdV"
MODEL = "gpt-5-mini"

PROJECT_NAME = "adam_and_eve"
EXPERIMENT_NAME = "initial"
TEXT_FOLDER = f"generated_text/{PROJECT_NAME}/{EXPERIMENT_NAME}/"
FILTERED_FOLDER = f"./filtered_text/{PROJECT_NAME.replace(' ', '-')}/{EXPERIMENT_NAME.replace(' ', '-')}/"

# create folders
try:
    os.mkdir(f"./filtered_text/{PROJECT_NAME.replace(' ', '-')}")
except FileExistsError: pass
try:
    os.mkdir(FILTERED_FOLDER)
except FileExistsError: pass


In [ ]:
def filter_and_save(model, folder) -> list[str]:
    prompt = ""
    with open(folder + "system_prompt.txt", 'r') as f:
        prompt += f.read() + "\n\n"
    with open(folder + "user_prompt.txt", 'r') as f:
        prompt += f.read()

    # setup client
    client = OpenAI(
        api_key=os.environ.get("OPENAI_API_KEY"),
    )
    chat_function = client.chat.completions.create
    extract_function = lambda r: r.choices[0].message.content
    
    # generate text and save to files
    files = [fn for fn in os.listdir(folder) if fn.replace(".", "").isdecimal()]

    all_text = ""

    for fn in tqdm(files):
        text = ""
        with open(folder + fn, 'r') as f:
            text = f.read()
        response = chat_function(
            model=model,
            messages=[{
                "role": "system",
                "content": "You are a science fiction editor/writer. You will be provided a story outline prompt, then some text written by a writer. You have to filter and return the good/interesting written text. Be selective, somewhere between 5-10% of text is usable, often the whole text is unusable."
            },{
                "role": "user",
                "content": prompt
            },{
                "role": "user",
                "content": text
            }],
            #temperature=temp
        )
        filtered_text = extract_function(response)
        all_text += (fn + "\n" + filtered_text + "\n\n---\n\n") if filtered_text else ""
    
    with open(FILTERED_FOLDER + str(datetime.datetime.now()), 'w') as f:
        f.write(all_text)


In [12]:
file_names = sorted(os.listdir(TEXT_FOLDER))
file_names

['.ipynb_checkpoints',
 '2025-02-02 10:06:21.895171',
 '2025-02-02 10:36:13.224987',
 '2025-02-02 10:38:05.308279',
 '2025-02-02 10:38:59.375849',
 '2025-02-02 10:40:59.200546',
 '2025-02-02 10:42:32.483271',
 '2025-02-02 10:43:39.922796',
 '2025-02-02 10:45:12.310464',
 '2025-02-02 10:52:51.196961',
 '2025-02-02 10:53:39.074500',
 '2025-02-02 10:55:38.979072',
 '2025-02-02 10:58:05.646643',
 '2025-02-02 10:59:30.719864',
 '2025-02-02 11:01:49.950718',
 '2025-02-02 11:03:59.768551',
 '2025-02-02 11:06:19.881430',
 '2025-02-02 11:08:03.867395',
 '2025-02-02 11:11:26.789073',
 '2025-02-02 11:13:29.421156',
 '2025-02-02 11:15:55.545324',
 '2025-02-02 11:17:50.316457',
 '2025-02-02 11:20:49.329630',
 '2025-02-02 11:22:30.402177',
 '2025-02-02 11:25:31.749131',
 '2025-02-02 22:45:46.296854',
 '2025-02-03 18:52:29.401501',
 '2025-02-03 18:56:05.232001',
 '2025-02-03 18:58:35.962751',
 '2025-02-03 19:11:42.352805',
 '2025-02-03 19:14:17.389578',
 '2025-02-03 19:16:12.067358',
 '2025-02-03 19:

In [13]:
# *** CHANGE THIS ***
selected_files = file_names[31:]

for folder in tqdm(selected_files):
    filter_and_save(MODEL, TEXT_FOLDER + folder + "/")

  0%|                                                                      | 0/23 [00:00<?, ?it/s]
%|                                                                      | 0/10 [00:00<?, ?it/s]
%|██████▏                                                       | 1/10 [00:08<01:13,  8.15s/it]
%|████████████▍                                                 | 2/10 [00:17<01:08,  8.58s/it]
%|██████████████████▌                                           | 3/10 [00:34<01:29, 12.74s/it]
%|████████████████████████▊                                     | 4/10 [00:44<01:08, 11.44s/it]
%|███████████████████████████████                               | 5/10 [00:57<00:59, 12.00s/it]
%|█████████████████████████████████████▏                        | 6/10 [01:02<00:38,  9.58s/it]
%|███████████████████████████████████████████▍                  | 7/10 [01:22<00:39, 13.09s/it]
%|█████████████████████████████████████████████████▌            | 8/10 [01:30<00:23, 11.59s/it]
%|███████████████████████████████████

In [14]:
len(selected_files)

23